In [ ]:
# ===================================================
# 1. Ask you to upload the CSV file.
# 2. Ask for your OpenAI API key.
# 3. Generate unique Belarusian QAs from the first 3 paragraphs.

!pip install openai pandas

import os, json
import pandas as pd
from openai import OpenAI
from google.colab import files

# --- Step 1: Upload CSV ---
print("📂 Please upload belwiki_500_paragraphs_topic_balanced.csv")
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

# --- Step 2: Ask for API key securely ---
import getpass
api_key = getpass.getpass("🔑 Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# --- Step 3: Load First 3 Paragraphs ---
df = pd.read_csv(csv_path)
paragraphs = df.iloc[:3, 0].dropna().tolist()

# --- Prompt Template ---
def make_prompt(context):
    return f"""
Ты — асцярожны пісьменнік па-беларуску.
Ствары адно ўнікальнае пытанне і кароткі фактычны адказ з гэтага параграфа.

Правілы:
- І пытанне, і адказ мусяць быць на беларускай мове.
- Адказ павінен дакладна паходзіць з параграфа, без дадатковай інфармацыі.
- Не капіюй тэкст цалкам: пытанне павінна быць сфармулявана нанова.
- Адказ павінен быць як мага карацейшы і дакладны.
- Вяртай строгі JSON фармат:
{{"qas":[{{"question":"...", "answer":"..."}}]}}

Параграф:
\"\"\"{context}\"\"\"
"""

#"""You are a careful Belarusian QA writer.
#Given the following Belarusian paragraph, write 2-3 question–answer pairs.
#Rules:
#- Both question and answer must be in Belarusian.
#- The answer must come only from the paragraph.
#- Keep answers short and factual.
#- Return strict JSON: {{"qas":[{{"question":"...","answer":"..."}}, ...]}}

#Paragraph:
#\"\"\"{context}\"\"\"



# --- Step 4: Generate QAs ---
results = []
for para in paragraphs:
    prompt = make_prompt(para)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    text = resp.choices[0].message.content.strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        data = {"qas":[]}

    results.append({"context": para, "qas": data.get("qas", [])})

# --- Step 5: Show Output ---
print("✅ Generated Belarusian QA pairs:\n")
print(json.dumps(results, ensure_ascii=False, indent=2))


📂 Please upload belwiki_500_paragraphs_topic_balanced.csv


Saving belwiki_500_paragraphs_topic_balanced.csv to belwiki_500_paragraphs_topic_balanced (3).csv
🔑 Enter your OpenAI API key: ··········


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
!pip install pandas nltk razdel

import pandas as pd
import nltk, random, json
from razdel import sentenize
from google.colab import files

nltk.download("punkt")

# --- Step 1: Upload CSV ---
print("📂 Please upload belwiki_500_paragraphs_topic_balanced.csv")
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

# --- Step 2: Load First 3 Paragraphs ---
df = pd.read_csv(csv_path)
paragraphs = df.iloc[:3, 0].dropna().tolist()

# --- Helper: Create QA ---
def make_qas(paragraph):
    sentences = [s.text for s in sentenize(paragraph)]
    qas = []

    for sent in sentences[:1]:  # take 1 key sentence per paragraph
        words = sent.split()
        if len(words) < 5:
            continue

        # Pick a random keyword (not стоп-слова)
        candidate_words = [w for w in words if w[0].isupper() or len(w) > 5]
        if not candidate_words:
            candidate_words = words[1:-1]

        if not candidate_words:
            continue

        answer = random.choice(candidate_words).strip(".,;:()")

        # Replace answer with "___" in question
        question = sent.replace(answer, "___", 1)
        qas.append({
            "question": f"Якое слова павінна стаяць на месцы ___ у сказе: «{question}»?",
            "answer": answer
        })

    return qas

# --- Step 3: Generate ---
results = []
for para in paragraphs:
    qas = make_qas(para)
    results.append({"context": para, "qas": qas})

# --- Step 4: Show Output ---
print("✅ Generated Belarusian QA pairs:\n")
print(json.dumps(results, ensure_ascii=False, indent=2))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


📂 Please upload belwiki_500_paragraphs_topic_balanced.csv


Saving belwiki_500_paragraphs_topic_balanced.csv to belwiki_500_paragraphs_topic_balanced.csv


TypeError: expected string or bytes-like object, got 'int'